In [1]:
import pandas as pd
from pathlib import Path
import ast
import json

# -------------------------------------------------------
# CONFIG
# -------------------------------------------------------
root = Path(r"G:\My Drive\PhD\proposal\code\chp2\api_convenios_2007_2026")
pattern = "api_*"
out_all = root / "convenios_compactados_todos.csv"
out_mcid = root / "convenios_compactados_MCID.csv"
out_mcid_parquet = root / "convenios_compactados_MCID.parquet"  # opcional


In [6]:

# -------------------------------------------------------
# Helpers
# -------------------------------------------------------
def parse_orgao_cell(x):
    """
    Converte a célula 'orgao' (string) em dict, se possível.
    Aceita:
      - string estilo python dict: "{'nome': ...}"
      - JSON: '{"nome": ...}'
      - já dict
    """
    if isinstance(x, dict):
        return x
    if pd.isna(x):
        return None
    s = str(x).strip()
    if not s:
        return None

    # Tenta python dict string
    try:
        return ast.literal_eval(s)
    except Exception:
        pass

    # Tenta JSON
    try:
        return json.loads(s)
    except Exception:
        return None

def is_mcid(orgao_dict):
    """
    Mantém se for MCID:
      orgao['sigla'] == 'MCID' OU
      orgao['codigoSIAFI'] == '56000' OU
      orgao['orgaoMaximo']['codigo'] == '56000'
    """
    if not isinstance(orgao_dict, dict):
        return False

    sigla = str(orgao_dict.get("sigla", "")).strip().upper()
    codigo = str(orgao_dict.get("codigoSIAFI", "")).strip()

    orgao_max = orgao_dict.get("orgaoMaximo", {})
    if isinstance(orgao_max, dict):
        codigo_max = str(orgao_max.get("codigo", "")).strip()
        sigla_max = str(orgao_max.get("sigla", "")).strip().upper()
    else:
        codigo_max, sigla_max = "", ""

    return (sigla == "MCID") or (codigo == "56000") or (codigo_max == "56000") or (sigla_max == "MCID")

def read_csv_robust(path):
    # tenta ; primeiro (muito comum no Brasil), depois ,
    for sep in [";", ","]:
        try:
            return pd.read_csv(
                path,
                sep=sep,
                dtype=str,
                engine="python",       # mais tolerante
                on_bad_lines="skip",   # pula linhas problemáticas
                encoding="utf-8"
            )
        except UnicodeDecodeError:
            return pd.read_csv(
                path,
                sep=sep,
                dtype=str,
                engine="python",
                on_bad_lines="skip",
                encoding="latin-1"
            )
        except Exception:
            pass
    # se falhar tudo:
    raise

# -------------------------------------------------------
# 1) Ler e compactar todos os CSVs
# -------------------------------------------------------
dfs = []
folders = sorted([p for p in root.glob(pattern) if p.is_dir()])

print(f"Pastas encontradas: {len(folders)}")

for folder in folders:
    csvs = list(folder.glob("*.csv"))
    if len(csvs) == 0:
        print(f"[WARN] Sem CSV em: {folder}")
        continue
    if len(csvs) > 1:
        print(f"[WARN] Mais de 1 CSV em {folder}. Vou pegar o primeiro: {csvs[0].name}")

    csv_path = csvs[0]

    df = read_csv_robust(csv_path)

    df["__source_file"] = str(csv_path)  # rastrear origem
    dfs.append(df)

if not dfs:
    raise RuntimeError("Nenhum CSV foi lido. Verifique o caminho/padrão.")

all_df = pd.concat(dfs, ignore_index=True)


Pastas encontradas: 451
[WARN] Sem CSV em: G:\My Drive\PhD\proposal\code\chp2\api_convenios_2007_2026\api_CE_2313500_trairi
[WARN] Sem CSV em: G:\My Drive\PhD\proposal\code\chp2\api_convenios_2007_2026\api_MA_2106409_mata_roma
[WARN] Sem CSV em: G:\My Drive\PhD\proposal\code\chp2\api_convenios_2007_2026\api_PA_1505437_ourilandia_do_norte
[WARN] Sem CSV em: G:\My Drive\PhD\proposal\code\chp2\api_convenios_2007_2026\api_PA_1506583_santa_maria_das_barreiras


In [ ]:
all_df



In [ ]:

# (Opcional) salvar tudo compactado
all_df.to_csv(out_all, index=False, encoding="utf-8")
print("✅ Salvei todos compactados em:", out_all)


In [7]:

# -------------------------------------------------------
# 2) Filtrar somente MCID (Ministério das Cidades)
# -------------------------------------------------------
# Parse da coluna orgao
all_df["__orgao_dict"] = all_df["orgao"].map(parse_orgao_cell)

# Filtrar
mcid_df = all_df[all_df["__orgao_dict"].map(is_mcid)].copy()

# Remover coluna auxiliar (ou deixar, se quiser auditar)
mcid_df = mcid_df.drop(columns=["__orgao_dict"])

# Salvar filtrado
mcid_df.to_csv(out_mcid, index=False, encoding="utf-8")
print("✅ Salvei filtrado MCID em:", out_mcid)

# Parquet (opcional, mais leve e rápido)
try:
    mcid_df.to_parquet(out_mcid_parquet, index=False)
    print("✅ Salvei também em Parquet:", out_mcid_parquet)
except Exception as e:
    print("[INFO] Não consegui salvar parquet (talvez falte pyarrow/fastparquet). Erro:", e)

print("Total linhas (tudo):", len(all_df))
print("Total linhas (MCID):", len(mcid_df))

✅ Salvei filtrado MCID em: G:\My Drive\PhD\proposal\code\chp2\api_convenios_2007_2026\convenios_compactados_MCID.csv
✅ Salvei também em Parquet: G:\My Drive\PhD\proposal\code\chp2\api_convenios_2007_2026\convenios_compactados_MCID.parquet
Total linhas (tudo): 27835
Total linhas (MCID): 6


In [ ]:
mcid_df